<a href="https://colab.research.google.com/github/manikanta-eng/block-chain-/blob/main/assignment_10_block_chain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

# ============================================================
# BLOCKCHAIN ENGINEERING PRACTICAL
# EXPLORING LAYER 2 - LOCAL BLOCKCHAIN SIMULATION
# Solidity + Web3.py + Ethereum Tester
# ============================================================

import sys
import subprocess
import time

# STEP 1: INSTALL LIBRARIES

print("Installing required libraries...")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "web3", "eth-tester[py-evm]", "py-solc-x"
])

from web3 import Web3, EthereumTesterProvider
from solcx import install_solc, compile_source

print("Libraries installed successfully!")

# STEP 2: CONNECT TO LOCAL BLOCKCHAIN

print("\nSTEP 1: BLOCKCHAIN CONNECTION")
print("--------------------------------")

w3 = Web3(EthereumTesterProvider())

if not w3.is_connected():
    raise Exception("Blockchain connection failed!")

account = w3.eth.accounts[0]

print("Network: Local Ethereum Test Blockchain")
print("Connection: SUCCESS")
print("Chain ID:", w3.eth.chain_id)
print("Test Account:", account)
print("Initial Balance:",
      w3.from_wei(w3.eth.get_balance(account), "ether"),
      "ETH")

# STEP 3: INSTALL SOLIDITY COMPILER

print("\nSTEP 2: SOLIDITY COMPILER")
print("--------------------------------")

install_solc("0.8.20")

print("Solidity Compiler: 0.8.20")
print("Installation: SUCCESS")

# STEP 4: CREATE SOLIDITY CONTRACT

solidity_code = """
// SPDX-License-Identifier: MIT
pragma solidity ^0.8.20;

contract Layer2Demo {

    uint256 public storedValue;
    string public storedMessage;

    // Event for value updates
    event ValueUpdated(uint256 value, address user);

    // Event for message updates
    event MessageUpdated(string message);

    // Store a number on the blockchain
    function setValue(uint256 _value) public {
        storedValue = _value;
        emit ValueUpdated(_value, msg.sender);
    }

    // Store a message on the blockchain
    function setMessage(string memory _message) public {
        storedMessage = _message;
        emit MessageUpdated(_message);
    }

    // Retrieve stored number
    function getValue() public view returns(uint256) {
        return storedValue;
    }

    // Retrieve stored message
    function getMessage() public view returns(string memory) {
        return storedMessage;
    }
}
"""

print("\nSTEP 3: SMART CONTRACT")
print("--------------------------------")
print("Contract created successfully!")

# STEP 5: COMPILE CONTRACT

compiled = compile_source(
    solidity_code,
    output_values=["abi", "bin"],
    solc_version="0.8.20"
)

contract_interface = compiled["<stdin>:Layer2Demo"]

abi = contract_interface["abi"]
bytecode = contract_interface["bin"]

print("Compilation: SUCCESS")
print("ABI generated: YES")
print("Bytecode generated: YES")

# STEP 6: DEPLOY CONTRACT

print("\nSTEP 4: CONTRACT DEPLOYMENT")
print("--------------------------------")

Contract = w3.eth.contract(
    abi=abi,
    bytecode=bytecode
)

start_time = time.time()

deployment_tx = Contract.constructor().transact({
    "from": account
})

deployment_receipt = w3.eth.wait_for_transaction_receipt(
    deployment_tx
)

deployment_time = time.time() - start_time

if deployment_receipt.status != 1:
    raise Exception("Deployment failed!")

contract_address = deployment_receipt.contractAddress

contract = w3.eth.contract(
    address=contract_address,
    abi=abi
)

print("Deployment: SUCCESS")
print("Contract Address:", contract_address)
print("Transaction Hash:", deployment_tx.hex())
print("Gas Used:", deployment_receipt.gasUsed)
print("Confirmation Time:",
      round(deployment_time, 4), "seconds")

# STEP 7: STORE VALUE

print("\nSTEP 5: STORE VALUE")
print("--------------------------------")

input_value = 100

start_time = time.time()

value_tx = contract.functions.setValue(
    input_value
).transact({"from": account})

value_receipt = w3.eth.wait_for_transaction_receipt(value_tx)

value_time = time.time() - start_time

if value_receipt.status != 1:
    raise Exception("Value storage failed!")

print("Input Value:", input_value)
print("Stored Value:",
      contract.functions.getValue().call())
print("Transaction: SUCCESS")
print("Gas Used:", value_receipt.gasUsed)
print("Execution Time:", round(value_time, 4), "seconds")

# STEP 8: STORE MESSAGE

print("\nSTEP 6: STORE MESSAGE")
print("--------------------------------")

input_message = "Hello Layer 2 Blockchain"

start_time = time.time()

message_tx = contract.functions.setMessage(
    input_message
).transact({"from": account})

message_receipt = w3.eth.wait_for_transaction_receipt(
    message_tx
)

message_time = time.time() - start_time

if message_receipt.status != 1:
    raise Exception("Message storage failed!")

print("Input Message:", input_message)
print("Stored Message:",
      contract.functions.getMessage().call())
print("Transaction: SUCCESS")
print("Gas Used:", message_receipt.gasUsed)
print("Execution Time:", round(message_time, 4), "seconds")

# STEP 9: READ BLOCKCHAIN DATA

print("\nSTEP 7: READ BLOCKCHAIN DATA")
print("--------------------------------")

stored_value = contract.functions.getValue().call()
stored_message = contract.functions.getMessage().call()

print("Retrieved Value:", stored_value)
print("Retrieved Message:", stored_message)
print("Data Retrieval: SUCCESS")

# STEP 10: GAS FEE CALCULATION

print("\nSTEP 8: GAS FEE ANALYSIS")
print("--------------------------------")

def calculate_fee(receipt):
    gas_price = receipt.effectiveGasPrice
    return receipt.gasUsed * gas_price

deployment_fee = calculate_fee(deployment_receipt)
value_fee = calculate_fee(value_receipt)
message_fee = calculate_fee(message_receipt)

total_fee = deployment_fee + value_fee + message_fee

print("Deployment Fee:",
      w3.from_wei(deployment_fee, "ether"), "ETH")

print("Value Transaction Fee:",
      w3.from_wei(value_fee, "ether"), "ETH")

print("Message Transaction Fee:",
      w3.from_wei(message_fee, "ether"), "ETH")

print("Total Fee:",
      w3.from_wei(total_fee, "ether"), "ETH")

# STEP 11: BLOCKCHAIN VERIFICATION

print("\nSTEP 9: BLOCKCHAIN VERIFICATION")
print("--------------------------------")

print("Latest Block:", w3.eth.block_number)
print("Contract Address:", contract_address)

assert stored_value == 100
assert stored_message == input_message

print("Value Verification: PASSED")
print("Message Verification: PASSED")

# STEP 12: FINAL RESULTS

print("\n==========================================")
print("       FINAL PRACTICAL RESULTS")
print("==========================================")

results = {
    "Blockchain Connection": "SUCCESS",
    "Solidity Compilation": "SUCCESS",
    "Contract Deployment": "SUCCESS",
    "Value Storage": "SUCCESS",
    "Message Storage": "SUCCESS",
    "Data Retrieval": "SUCCESS",
    "Gas Calculation": "SUCCESS",
    "Blockchain Verification": "SUCCESS"
}

for test, result in results.items():
    print(f"{test:30}: {result}")

print("------------------------------------------")
print("Input Value:", input_value)
print("Output Value:", stored_value)
print("Input Message:", input_message)
print("Output Message:", stored_message)
print("Total Gas Used:",
      deployment_receipt.gasUsed +
      value_receipt.gasUsed +
      message_receipt.gasUsed)

print("------------------------------------------")
print("ALL LOCAL BLOCKCHAIN TESTS PASSED!")
print("==========================================")


Installing required libraries...
Libraries installed successfully!

STEP 1: BLOCKCHAIN CONNECTION
--------------------------------
Network: Local Ethereum Test Blockchain
Connection: SUCCESS
Chain ID: 131277322940537
Test Account: 0x7E5F4552091A69125d5DfCb7b8C2659029395Bdf
Initial Balance: 1000000 ETH

STEP 2: SOLIDITY COMPILER
--------------------------------


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Solidity Compiler: 0.8.20
Installation: SUCCESS

STEP 3: SMART CONTRACT
--------------------------------
Contract created successfully!
Compilation: SUCCESS
ABI generated: YES
Bytecode generated: YES

STEP 4: CONTRACT DEPLOYMENT
--------------------------------
Deployment: SUCCESS
Contract Address: 0xF2E246BB76DF876Cef8b38ae84130F4F55De395b
Transaction Hash: 995ec2d4b32a8705d10750d1aef0b982090730d2f0dc84eb0a256a7706f67957
Gas Used: 544275
Confirmation Time: 0.0427 seconds

STEP 5: STORE VALUE
--------------------------------
Input Value: 100
Stored Value: 100
Transaction: SUCCESS
Gas Used: 45333
Execution Time: 0.0518 seconds

STEP 6: STORE MESSAGE
--------------------------------
Input Message: Hello Layer 2 Blockchain
Stored Message: Hello Layer 2 Blockchain
Transaction: SUCCESS
Gas Used: 47441
Execution Time: 0.0602 seconds

STEP 7: READ BLOCKCHAIN DATA
--------------------------------
Retrieved Value: 100
Retrieved Message: Hello Layer 2 Blockchain
Data Retrieval: SUCCESS

STEP 8: 